# Qwen3-4B Multi-Agent Test Environment

## Setup
- Cell 1: pip install + 코드 전체 + 모델 로드 (최초 1회)
- Cell 2: 에이전트 생성 + 프롬프트 정의
- Cell 3+: 실험 (복사해서 쓰면 됨)

---

## API Reference

### Agent 생성
```python
agent = Agent("이름", "system prompt")
```

### 단일 추론
```python
# 기본 (thinking OFF, 256 tokens)
r = agent.say("질문")

# thinking ON + 토큰 늘리기
r = agent.say("질문", max_tokens=2048, thinking=True)

# 결과 접근
r["response"]   # 응답 텍스트
r["tokens"]     # 생성 토큰 수
r["time"]       # 소요 시간(초)
```

### 파라미터
| 파라미터 | 기본값 | 설명 |
|----------|--------|------|
| `max_tokens` | 256 | 생성 최대 토큰. thinking=True면 2048 권장 |
| `thinking` | False | True: Qwen3 내부 추론(`<think>`) 활성화. 느리지만 정확도↑ |

### 프롬프트 변경
```python
agent.set_prompt("새 system prompt")
```

### A → B 통신
```python
result = send(a, b, "메시지")
result["sender"]["response"]    # A의 응답
result["receiver"]["response"]  # B의 응답
result["total_tokens"]          # 총 토큰
```

### A → B → C 체인
```python
result = chain([a, b, c], "메시지")
result["final"]          # 마지막 에이전트 응답
result["total_tokens"]   # 총 토큰
result["steps"]          # 각 단계별 결과 리스트
```

### ask() 헬퍼 (ABCD 문제용)
```python
r = ask(agent, MATH_PROMPT, "질문텍스트", max_tokens=512)
r["answer"]     # 추출된 답 (A/B/C/D 또는 N/A)
r["response"]   # 전체 응답
r["tokens"]     # 토큰 수
r["time"]       # 소요 시간
```

### 유틸리티
```python
extract_number("답은 42입니다")   # → 42.0
grade(got, expected)              # 채점 (10% 오차 허용)
```

In [ ]:
# ============================================================
# Cell 1: 모델 로드 (최초 1회)
# ============================================================
!pip install -q torch transformers accelerate

import torch
import time
import re

# Global model/tokenizer
_model = None
_tokenizer = None
_device = None

def load_model(model_id: str = "Qwen/Qwen3-4B"):
    global _model, _tokenizer, _device
    from transformers import AutoModelForCausalLM, AutoTokenizer
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if _device == "cuda" else torch.float32
    print(f"Device: {_device}")
    if _device == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Loading {model_id}...")
    t0 = time.time()
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype,
        device_map="auto" if _device == "cuda" else None,
    )
    if _device == "cpu":
        _model = _model.to(_device)
    params = sum(p.numel() for p in _model.parameters()) / 1e9
    print(f"Loaded in {time.time()-t0:.1f}s ({params:.1f}B params)")

class Agent:
    def __init__(self, name: str, system_prompt: str):
        self.name = name
        self.system_prompt = system_prompt
        self.history = []

    def say(self, message: str, max_tokens: int = 256, thinking: bool = False) -> dict:
        if _model is None:
            raise RuntimeError("load_model()을 먼저 실행하세요.")
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": message},
        ]
        text = _tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=thinking
        )
        inputs = _tokenizer(text, return_tensors="pt").to(_model.device)
        t0 = time.time()
        with torch.no_grad():
            output = _model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        elapsed = time.time() - t0
        response = _tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        gen_tokens = output.shape[1] - inputs["input_ids"].shape[1]
        result = {"response": response, "tokens": gen_tokens, "time": round(elapsed, 1)}
        self.history.append({"input": message, **result})
        return result

    def set_prompt(self, new_prompt: str):
        self.system_prompt = new_prompt
        return self

    def __repr__(self):
        return f"Agent('{self.name}')"

def send(sender, receiver, message, max_tokens=256, verbose=True):
    s = sender.say(message, max_tokens=max_tokens)
    r = receiver.say(s["response"], max_tokens=max_tokens)
    if verbose:
        print(f"[{sender.name}] {s['tokens']}tok, {s['time']}s")
        print(f"  >> {s['response'][:200]}")
        print(f"[{receiver.name}] {r['tokens']}tok, {r['time']}s")
        print(f"  >> {r['response'][:200]}")
    return {"sender": s, "receiver": r, "tx_tokens": s["tokens"], "total_tokens": s["tokens"] + r["tokens"]}

def chain(agents, message, max_tokens=256, verbose=True):
    current = message
    results = []
    for agent in agents:
        r = agent.say(current, max_tokens=max_tokens)
        results.append({"agent": agent.name, **r})
        if verbose:
            print(f"[{agent.name}] {r['tokens']}tok, {r['time']}s")
            print(f"  >> {r['response'][:200]}")
        current = r["response"]
    return {"steps": results, "final": results[-1]["response"], "total_tokens": sum(r["tokens"] for r in results)}

def extract_number(text):
    nums = re.findall(r'-?[\d,]+\.?\d*', text.replace(',', ''))
    return float(nums[0]) if nums else -999

def grade(got, expected, tolerance=0.1):
    if expected == 0:
        return abs(got) < tolerance
    return abs(got - expected) / abs(expected) < tolerance

load_model("Qwen/Qwen3-4B")

In [ ]:
# ============================================================
# Cell 2: 에이전트 생성 (Math / Science / Code)
# ============================================================
SYSTEM_PROMPT = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."

MATH_PROMPT = (
    "You are a math agent. Given the input question, reason step-by-step "
    "and put the final answer inside \\boxed{YOUR_FINAL_ANSWER}. "
    "Your final answer must be selected from A,B,C,D. "
    "For example \\boxed{A}. Do not add any other contents inside the box."
)

SCIENCE_PROMPT = (
    "You are a science agent. Given the input question, reason step-by-step "
    "and put the final answer inside \\boxed{YOUR_FINAL_ANSWER}. "
    "Your final answer must be selected from A,B,C,D. "
    "For example \\boxed{A}. Do not add any other contents inside the box."
)

CODE_PROMPT = (
    "You are a code agent. Given the input question, reason step-by-step "
    "and put the final answer inside \\boxed{YOUR_FINAL_ANSWER}. "
    "Your final answer must be selected from A,B,C,D. "
    "For example \\boxed{A}. Do not add any other contents inside the box."
)

math_agent = Agent("Math", SYSTEM_PROMPT)
science_agent = Agent("Science", SYSTEM_PROMPT)
code_agent = Agent("Code", SYSTEM_PROMPT)

AGENTS = {"Math": math_agent, "Science": science_agent, "Code": code_agent}
PROMPTS = {"Math": MATH_PROMPT, "Science": SCIENCE_PROMPT, "Code": CODE_PROMPT}

def ask(agent, prompt_template, question, max_tokens=512):
    """프롬프트 템플릿 + 질문으로 추론 실행."""
    full_prompt = f"{prompt_template}\nQuestion: {question}\nYour response:"
    r = agent.say(full_prompt, max_tokens=max_tokens)
    import re
    match = re.search(r'\\boxed\{([A-D])\}', r["response"])
    answer = match.group(1) if match else "N/A"
    print(f"[{agent.name}] {r['tokens']}tok, {r['time']}s")
    print(f"  Answer: {answer}")
    print(f"  Response: {r['response'][:300]}")
    return {"answer": answer, **r}

print(math_agent, science_agent, code_agent)

In [ ]:
# ---- Math Agent: 단일 문제 ----
q1 = """If f(x) = 2x^2 - 3x + 1, what is f(3)?
A) 10
B) 12
C) 14
D) 16"""

r1 = ask(math_agent, MATH_PROMPT, q1)

In [ ]:
# ---- Math Agent: 확률 문제 ----
q2 = """A bag contains 3 red and 5 blue balls. Two balls are drawn without replacement.
What is the probability that both are red?
A) 3/28
B) 3/8
C) 9/64
D) 1/7"""

r2 = ask(math_agent, MATH_PROMPT, q2)

In [ ]:
# ---- Science Agent: 물리 문제 ----
q3 = """An object is dropped from rest. How far does it fall in the first 3 seconds?
(Use g = 10 m/s^2)
A) 15 m
B) 30 m
C) 45 m
D) 90 m"""

r3 = ask(science_agent, SCIENCE_PROMPT, q3)

In [ ]:
# ---- Science Agent: 화학 문제 ----
q4 = """What is the molecular formula of glucose?
A) C6H10O5
B) C6H12O6
C) C12H22O11
D) CH3COOH"""

r4 = ask(science_agent, SCIENCE_PROMPT, q4)

In [ ]:
# ---- 결과 요약 ----
results = [
    ("Math - f(3)", r1["answer"], "A"),       # 2(9)-3(3)+1 = 10
    ("Math - Prob", r2["answer"], "A"),       # 3/8 * 2/7 = 3/28
    ("Science - Physics", r3["answer"], "C"), # 0.5*10*9 = 45m
    ("Science - Chem", r4["answer"], "B"),    # C6H12O6
]

print("=" * 40)
print(f"{'Problem':<22} {'Got':<5} {'Exp':<5} {'Result'}")
print("-" * 40)
correct = 0
for name, got, expected in results:
    ok = got == expected
    correct += ok
    print(f"{name:<22} {got:<5} {expected:<5} {'OK' if ok else 'FAIL'}")
print("-" * 40)
print(f"Accuracy: {correct}/{len(results)} ({correct/len(results)*100:.0f}%)")

---
# Key Idea 1: Mutual Cognitive Context Inference
- **Tx**: `math_agent` (고정), **Rx**: `science_agent` (고정)
- 조건 차이 = 유저 메시지에 상대 에이전트 정보를 **추가하느냐 마느냐**

| Condition | Tx 메시지에 Rx 정보 | Rx 메시지에 Tx 정보 |
|-----------|:-:|:-:|
| **No Context** | X | X |
| **One-way (Tx→Rx)** | O | X |
| **One-way (Rx→Tx)** | X | O |
| **Mutual** | O | O |

In [ ]:
# ============================================================
# Key Idea 1: 실험 설정
# ============================================================
DATA = "revenue=10M, cost=6M, employees=500, debt=2M, equity=8M, cash=1M, tax=25%"
TASK = "debt-to-equity ratio (debt / equity)"
EXPECTED = 0.25

# Tx(math_agent)에게 보내는 메시지 변형
TX_BASE = f"Data: {DATA}\nSummarize this data."
TX_KNOWS_RX = f"Data: {DATA}\nThe receiver is a science agent that needs debt and equity to compute a ratio. Send only relevant key=value pairs."

# Rx(science_agent)에게 보내는 메시지 변형 (tx_out이 앞에 붙음)
RX_BASE = f"Task: {TASK}\nCompute the value. Output only a number."
RX_KNOWS_TX = f"Task: {TASK}\nThe sender is a math agent who provides structured numerical data. Extract the numbers and compute. Output only a number."

# 4개 조건
conditions = {
    "no_context":        {"tx_msg": TX_BASE,     "rx_extra": RX_BASE},
    "oneway_tx_knows_rx": {"tx_msg": TX_KNOWS_RX, "rx_extra": RX_BASE},
    "oneway_rx_knows_tx": {"tx_msg": TX_BASE,     "rx_extra": RX_KNOWS_TX},
    "mutual":            {"tx_msg": TX_KNOWS_RX, "rx_extra": RX_KNOWS_TX},
}

print("Agents: math_agent(Tx), science_agent(Rx)")
print("Conditions:", list(conditions.keys()))

In [ ]:
# ============================================================
# Key Idea 1: 4조건 실행
# ============================================================
exp_results = {}

for cond_name, cfg in conditions.items():
    print(f"\n{'='*50}")
    print(f"  {cond_name}")
    print(f"{'='*50}")

    # Tx: math_agent (고정)
    tx_out = math_agent.say(cfg["tx_msg"])
    print(f"[MathAgent→Tx] {tx_out['tokens']}tok, {tx_out['time']}s")
    print(f"  >> {tx_out['response'][:200]}")

    # Rx: science_agent (고정)
    rx_msg = f"Info: {tx_out['response']}\n{cfg['rx_extra']}"
    rx_out = science_agent.say(rx_msg)
    print(f"[ScienceAgent→Rx] {rx_out['tokens']}tok, {rx_out['time']}s")
    print(f"  >> {rx_out['response'][:200]}")

    got = extract_number(rx_out["response"])
    ok = grade(got, EXPECTED)
    print(f"  Answer: {got} (expected {EXPECTED}) {'OK' if ok else 'FAIL'}")

    exp_results[cond_name] = {
        "tx_tokens": tx_out["tokens"],
        "rx_tokens": rx_out["tokens"],
        "total_tokens": tx_out["tokens"] + rx_out["tokens"],
        "total_time": tx_out["time"] + rx_out["time"],
        "answer": got, "correct": ok,
    }

In [ ]:
# ============================================================
# Key Idea 1: 결과 비교표
# ============================================================
print(f"{'Condition':<22} {'Tx':<8} {'Rx':<8} {'Total':<8} {'Time':<8} {'Ans':<8} {'OK?'}")
print("-" * 65)
for cond, r in exp_results.items():
    print(f"{cond:<22} {r['tx_tokens']:<8} {r['rx_tokens']:<8} {r['total_tokens']:<8} "
          f"{r['total_time']:<8} {r['answer']:<8} {'OK' if r['correct'] else 'FAIL'}")

if exp_results['no_context']['tx_tokens'] > 0:
    reduction = (1 - exp_results['mutual']['tx_tokens'] / exp_results['no_context']['tx_tokens']) * 100
    print(f"\nToken reduction (no_context → mutual): {reduction:.1f}%")

---
# Key Idea 2: Stage-Wise CoT Switching
- **Tx**: `math_agent` (고정), **Rx**: `science_agent` (고정)
- Tx가 3단계 CoT로 추론 (이해→추론→전달), 단계별 메시지만 변경
- Rx는 1단계(일반) 또는 3단계(수신→계산→출력)

| Condition | Tx S1-S2 메시지 | Tx S3 메시지 | Rx |
|-----------|:---:|:---:|:---:|
| **All General** | 일반 | 일반 | 1단계 |
| **All Aware** | Rx 인지 | Rx 인지 | 1단계 |
| **Tx Only Switch** | 일반 | Rx 인지 | 1단계 |
| **Both Switch** | 일반 | Rx 인지 | 3단계 (Tx 인지) |

In [ ]:
# ============================================================
# Key Idea 2: 실험 설정
# ============================================================
DATA2 = "revenue=10M, cost=6M, employees=500, debt=2M, equity=8M, cash=1M, tax=25%"
TASK2 = "debt-to-equity ratio (debt / equity)"
EXPECTED2 = 0.25

# --- Tx 단계별 메시지 (math_agent에게 보냄) ---
# General
TX_S1_GEN = "Parse this data and list all key-value pairs.\nData: {data}"
TX_S2_GEN = "Based on this, identify what financial metrics could be computed.\n{prev}"
TX_S3_GEN = "Summarize into a concise message.\n{prev}"

# Audience-Aware
TX_S1_AWR = "Parse this data, focusing on ratio components.\nData: {data}"
TX_S2_AWR = "The receiver is a science agent that needs debt-to-equity ratio. Identify only debt and equity.\n{prev}"
TX_S3_AWR = "Send only debt and equity as key=value pairs for the science agent. Nothing else.\n{prev}"

# --- Rx 메시지 (science_agent에게 보냄) ---
# 1단계
RX_1STAGE = "Info: {tx_out}\nTask: {task}\nCompute the value. Output only a number."

# 3단계 (Both Switch용)
RX_S1 = "The math agent sent you this data. Parse the key=value pairs.\nInfo: {tx_out}"
RX_S2 = "From the parsed values, compute {task}.\n{prev}"
RX_S3 = "Output only the final number. Nothing else.\n{prev}"

# --- 4개 조건 ---
cot_conditions = {
    "all_general": {
        "tx": [TX_S1_GEN, TX_S2_GEN, TX_S3_GEN],
        "rx": [RX_1STAGE],
    },
    "all_aware": {
        "tx": [TX_S1_AWR, TX_S2_AWR, TX_S3_AWR],
        "rx": [RX_1STAGE],
    },
    "tx_only_switch": {
        "tx": [TX_S1_GEN, TX_S2_GEN, TX_S3_AWR],
        "rx": [RX_1STAGE],
    },
    "both_switch": {
        "tx": [TX_S1_GEN, TX_S2_GEN, TX_S3_AWR],
        "rx": [RX_S1, RX_S2, RX_S3],
    },
}

print("Agents: math_agent(Tx CoT), science_agent(Rx)")
print("Conditions:", list(cot_conditions.keys()))

In [ ]:
# ============================================================
# Key Idea 2: 실행
# ============================================================
cot_results = {}

for cond_name, cfg in cot_conditions.items():
    print(f"\n{'='*55}")
    print(f"  {cond_name}  (Tx {len(cfg['tx'])} stages, Rx {len(cfg['rx'])} stages)")
    print(f"{'='*55}")

    # --- Tx CoT: math_agent 고정 ---
    stage_names = ["이해", "추론", "전달"]
    prev = ""
    tx_results = []
    for i, tmpl in enumerate(cfg["tx"]):
        msg = tmpl.format(data=DATA2, prev=prev)
        r = math_agent.say(msg)
        tx_results.append(r)
        print(f"  [Tx-{stage_names[i]}] {r['tokens']}tok, {r['time']}s")
        print(f"    >> {r['response'][:150]}")
        prev = r["response"]

    tx_message = prev  # Stage 3 출력 = 전송 메시지

    # --- Rx: science_agent 고정 ---
    rx_stage_names = ["수신", "계산", "출력"]
    prev = ""
    rx_results = []
    for i, tmpl in enumerate(cfg["rx"]):
        msg = tmpl.format(tx_out=tx_message, task=TASK2, prev=prev)
        r = science_agent.say(msg)
        rx_results.append(r)
        label = rx_stage_names[i] if len(cfg["rx"]) > 1 else "Rx"
        print(f"  [{label}] {r['tokens']}tok, {r['time']}s")
        print(f"    >> {r['response'][:150]}")
        prev = r["response"]

    got = extract_number(rx_results[-1]["response"])
    ok = grade(got, EXPECTED2)
    print(f"\n  >> Answer: {got} (expected {EXPECTED2}) {'OK' if ok else 'FAIL'}")

    cot_results[cond_name] = {
        "tx_msg_tokens": tx_results[-1]["tokens"],
        "tx_cot_tokens": sum(r["tokens"] for r in tx_results),
        "rx_tokens": sum(r["tokens"] for r in rx_results),
        "total_tokens": sum(r["tokens"] for r in tx_results) + sum(r["tokens"] for r in rx_results),
        "total_time": round(sum(r["time"] for r in tx_results) + sum(r["time"] for r in rx_results), 1),
        "answer": got, "correct": ok,
    }

In [ ]:
# ============================================================
# Key Idea 2: 결과 비교표
# ============================================================
print(f"{'Condition':<18} {'Tx msg':<8} {'CoT':<8} {'Rx':<8} {'Total':<8} {'Time':<8} {'Ans':<8} {'OK?'}")
print("-" * 75)
for cond, r in cot_results.items():
    print(f"{cond:<18} {r['tx_msg_tokens']:<8} {r['tx_cot_tokens']:<8} {r['rx_tokens']:<8} "
          f"{r['total_tokens']:<8} {r['total_time']:<8} {r['answer']:<8} {'OK' if r['correct'] else 'FAIL'}")

print(f"\n  All General  tx_msg: {cot_results['all_general']['tx_msg_tokens']}tok")
print(f"  Tx Switch    tx_msg: {cot_results['tx_only_switch']['tx_msg_tokens']}tok")
print(f"  Both Switch  tx_msg: {cot_results['both_switch']['tx_msg_tokens']}tok")

---
# Key Idea 4: Task Scheduling (Fixed-Order Chain)
- **3 에이전트**: `Math`, `Science`, `Code` (고정)
- A→B→C 체인, 모든 순열(3! = 6가지) 테스트
- 각 에이전트는 자기 프롬프트(MATH/SCIENCE/CODE_PROMPT)로 처리
- 최종 에이전트 C의 `\boxed{}` 답을 score로 채점
- 채널 latency / 전문 분야 매칭은 나중에 데이터 조작으로 반영

In [ ]:
# ============================================================
# Key Idea 4: 실험 설정
# ============================================================
from itertools import permutations

# 테스트 문제 (나중에 데이터셋으로 교체)
QUESTIONS_4 = [
    {
        "question": (
            "A car accelerates from rest at 2 m/s^2 for 5 seconds, then travels at constant speed. "
            "What is the total distance covered in the first 10 seconds?\n"
            "A) 50 m\nB) 75 m\nC) 100 m\nD) 125 m"
        ),
        "answer": "B",  # d1=0.5*2*25=25m, v=10m/s, d2=10*5=50m, total=75m
    },
    {
        "question": (
            "A function f(x) = x^3 - 6x^2 + 9x + 1. At which x does f(x) have a local maximum?\n"
            "A) x=0\nB) x=1\nC) x=3\nD) x=5"
        ),
        "answer": "B",  # f'=3x^2-12x+9=3(x-1)(x-3), local max at x=1
    },
    {
        "question": (
            "What is the output of: [x**2 for x in range(5) if x % 2 == 0]?\n"
            "A) [0, 4, 16]\nB) [1, 9, 25]\nC) [0, 2, 4]\nD) [4, 16]"
        ),
        "answer": "A",  # x=0,2,4 → 0,4,16
    },
]

# 모든 순열
agent_names = ["Math", "Science", "Code"]
all_orders = list(permutations(agent_names))
print(f"Questions: {len(QUESTIONS_4)}")
print(f"Orders ({len(all_orders)}):")
for i, order in enumerate(all_orders):
    print(f"  {i+1}. {' → '.join(order)}")

In [ ]:
# ============================================================
# Key Idea 4: 전순열 체인 실행
# ============================================================
import re as _re

def run_chain_abcd(order, question, max_tokens=512, verbose=True):
    """3-agent chain 실행, 최종 \\boxed{} 답 추출."""
    agents = [AGENTS[name] for name in order]
    prompts = [PROMPTS[name] for name in order]

    current = f"{prompts[0]}\nQuestion: {question}\nYour response:"
    results = []

    for i, agent in enumerate(agents):
        if i == 0:
            msg = current
        else:
            msg = f"{prompts[i]}\nPrevious agent's analysis:\n{current}\nYour response:"
        r = agent.say(msg, max_tokens=max_tokens)
        results.append({"agent": order[i], **r})
        if verbose:
            print(f"  [{order[i]}] {r['tokens']}tok, {r['time']}s")
            print(f"    >> {r['response'][:150]}")
        current = r["response"]

    # 최종 답 추출
    match = _re.search(r'\\boxed\{([A-D])\}', results[-1]["response"])
    answer = match.group(1) if match else "N/A"
    total_tokens = sum(r["tokens"] for r in results)
    total_time = round(sum(r["time"] for r in results), 1)

    return {
        "answer": answer,
        "total_tokens": total_tokens,
        "total_time": total_time,
        "steps": results,
    }

# --- 실행 ---
sched_results = {}

for order in all_orders:
    order_key = "→".join(order)
    scores = []
    total_tok = 0
    total_t = 0

    print(f"\n{'='*60}")
    print(f"  Order: {order_key}")
    print(f"{'='*60}")

    for qi, q in enumerate(QUESTIONS_4):
        print(f"\n  Q{qi+1}: {q['question'][:60]}...")
        r = run_chain_abcd(order, q["question"])
        correct = r["answer"] == q["answer"]
        scores.append(correct)
        total_tok += r["total_tokens"]
        total_t += r["total_time"]
        print(f"  >> Got: {r['answer']}, Expected: {q['answer']} {'OK' if correct else 'FAIL'}")

    accuracy = sum(scores) / len(scores)
    sched_results[order_key] = {
        "accuracy": accuracy,
        "correct": sum(scores),
        "total": len(scores),
        "total_tokens": total_tok,
        "total_time": round(total_t, 1),
        "details": scores,
    }

print("\n\nDone. All orders tested.")

In [ ]:
# ============================================================
# Key Idea 4: 결과 비교표
# ============================================================
print(f"{'Order':<25} {'Score':<12} {'Tokens':<10} {'Time':<8} {'Q1':<5} {'Q2':<5} {'Q3':<5}")
print("-" * 75)

# accuracy 기준 정렬
sorted_results = sorted(sched_results.items(), key=lambda x: -x[1]["accuracy"])

for order_key, r in sorted_results:
    q_marks = ["O" if d else "X" for d in r["details"]]
    print(f"{order_key:<25} {r['correct']}/{r['total']} ({r['accuracy']*100:.0f}%)   "
          f"{r['total_tokens']:<10} {r['total_time']:<8} {'  '.join(q_marks)}")

print(f"\n{'='*75}")
best = sorted_results[0]
worst = sorted_results[-1]
print(f"Best:  {best[0]} ({best[1]['accuracy']*100:.0f}%)")
print(f"Worst: {worst[0]} ({worst[1]['accuracy']*100:.0f}%)")